# Experiment: Lattice Spacing in Edges vs Nodes

Tests whether lattice spacing `a` belongs in edge features (as displacement
vectors `a·ê_μ`) rather than spacetime node features, and whether the
explicit `a^d` volume factor in the energy readout can be dropped.

**4 architecture variants:**

| Variant | `a` location | Volume scaling | Description |
|---------|-------------|----------------|-------------|
| `baseline` | nodes | `× a^d` | Current architecture (control) |
| `a_edges_vol` | edges | `× a^d` | Displacement edges, keep volume factor |
| `a_edges_no_vol` | edges | none | Displacement edges, drop volume factor |
| `a_nodes_no_vol` | nodes | none | Current nodes, drop volume factor |

**Two phases:**
- **Phase A** (spacing=1.0): Structural test using existing MC data
- **Phase B** (spacing=1/L): Scaling test with rescaled targets

**Runtime:** ~20–30 min on Colab T4 for all 32 runs (4 variants × 4 sizes × 2 phases).

## 0. Setup: Mount Drive & Install Dependencies

In [ ]:
import os
import sys

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/qft_graph'
    !pip install -q torch-geometric omegaconf
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
    os.chdir(PROJECT_ROOT)
    print(f'Working directory: {os.getcwd()}')
else:
    PROJECT_ROOT = os.path.abspath('..')
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
    os.chdir(PROJECT_ROOT)

print('Setup complete.')

In [ ]:
import json
import time
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch_geometric.loader import DataLoader as PyGDataLoader

from qft_graph.config import LatticeConfig, ModelConfig, TrainingConfig
from qft_graph.lattice.hypercubic import HypercubicLattice
from qft_graph.fields.scalar import ScalarField
from qft_graph.graphs.builder import HeteroGraphBuilder
from qft_graph.models.hetero_gnn import HeteroGNN
from qft_graph.training.losses import EnergyMatchingLoss
from qft_graph.training.metrics import energy_correlation, relative_error
from qft_graph.utils.reproducibility import set_seed

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

plt.style.use('dark_background')
%matplotlib inline

## 1. Experiment Configuration

In [ ]:
# === Experiment parameters ===
VARIANTS = {
    'baseline':       {'a_in_edges': False, 'volume_scaling': True},
    'a_edges_vol':    {'a_in_edges': True,  'volume_scaling': True},
    'a_edges_no_vol': {'a_in_edges': True,  'volume_scaling': False},
    'a_nodes_no_vol': {'a_in_edges': False, 'volume_scaling': False},
}

LATTICE_SIZES = [8, 16, 32, 64]
MASS_SQUARED = -0.5
COUPLING = 0.5
EPOCHS = 200
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
SEED = 42

DATA_DIR = Path('data/mc_configs')
OUTPUT_BASE = Path('experiments/runs/displacement_edges')

print(f'Variants: {list(VARIANTS.keys())}')
print(f'Lattice sizes: {LATTICE_SIZES}')
print(f'Epochs: {EPOCHS}')
print(f'Total runs per phase: {len(VARIANTS) * len(LATTICE_SIZES)}')

## 2. Training Function

In [ ]:
def train_variant(
    variant_name: str,
    variant_flags: dict,
    lattice_size: int,
    spacing: float,
    action_scale: float,
    phase: str,
) -> dict:
    """Train one variant at one lattice size and return metrics."""
    set_seed(SEED)

    # Load MC data
    data_path = DATA_DIR / f'phi4_{lattice_size}x{lattice_size}_m2={MASS_SQUARED}_lam={COUPLING}' / 'mc_data.pt'
    if not data_path.exists():
        print(f'  SKIPPED: {data_path} not found')
        return {'status': 'skipped', 'variant': variant_name, 'lattice_size': lattice_size, 'phase': phase}

    mc_data = torch.load(data_path, weights_only=False)
    configurations = mc_data['configurations']
    actions = mc_data['actions'] * action_scale

    # Build lattice and graphs
    lattice_config = LatticeConfig(dimensions=(lattice_size, lattice_size), spacing=spacing)
    lattice = HypercubicLattice(lattice_config)
    scalar_field = ScalarField()
    builder = HeteroGraphBuilder(lattice, [scalar_field], a_in_edges=variant_flags['a_in_edges'])

    dataset = builder.build_dataset(
        configurations={'scalar': configurations},
        actions=actions,
    )

    # Train/val split (80/20)
    n_train = int(0.8 * len(dataset))
    train_loader = PyGDataLoader(dataset[:n_train], batch_size=BATCH_SIZE, shuffle=True)
    val_loader = PyGDataLoader(dataset[n_train:], batch_size=BATCH_SIZE, shuffle=False)

    # Create model
    model_config = ModelConfig(
        a_in_edges=variant_flags['a_in_edges'],
        volume_scaling=variant_flags['volume_scaling'],
    )
    model = HeteroGNN(
        config=model_config,
        lattice_dim=lattice.dimension(),
        field_types={'scalar': scalar_field.dof_per_site()},
        lattice_spacing=lattice.lattice_spacing(),
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters())

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = EnergyMatchingLoss()

    print(f'  {variant_name} | {lattice_size}x{lattice_size} | spacing={spacing:.4f} | params={n_params}')

    start = time.time()
    best_corr = 0.0

    for epoch in range(1, EPOCHS + 1):
        # Train
        model.train()
        total_loss, n_batches = 0.0, 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            output = model(batch)
            loss = criterion(output['energy'], batch.y.to(device))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        scheduler.step()

        # Validate every 50 epochs + last epoch
        if epoch % 50 == 0 or epoch == EPOCHS:
            model.eval()
            all_pred, all_true = [], []
            with torch.no_grad():
                for batch in val_loader:
                    batch = batch.to(device)
                    output = model(batch)
                    all_pred.append(output['energy'].cpu())
                    all_true.append(batch.y.cpu())
            preds = torch.cat(all_pred)
            trues = torch.cat(all_true)
            corr = energy_correlation(preds, trues)
            rel_err = relative_error(preds, trues)
            best_corr = max(best_corr, corr)
            print(f'    Epoch {epoch:>3d}/{EPOCHS} | loss={total_loss/n_batches:.6f} | r={corr:.6f} | rel_err={rel_err:.6f}')

    elapsed = time.time() - start

    # Final evaluation
    model.eval()
    all_pred, all_true = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            output = model(batch)
            all_pred.append(output['energy'].cpu())
            all_true.append(batch.y.cpu())
    preds = torch.cat(all_pred)
    trues = torch.cat(all_true)
    final_corr = energy_correlation(preds, trues)
    final_rel_err = relative_error(preds, trues)

    # Save checkpoint
    run_dir = OUTPUT_BASE / f'phase_{phase}' / f'{variant_name}_{lattice_size}x{lattice_size}'
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), run_dir / 'model_final.pt')

    result = {
        'variant': variant_name,
        'lattice_size': lattice_size,
        'spacing': spacing,
        'action_scale': action_scale,
        'phase': phase,
        'n_params': n_params,
        'final_val_corr': round(final_corr, 6),
        'final_rel_err': round(final_rel_err, 6),
        'best_val_corr': round(best_corr, 6),
        'training_time_s': round(elapsed, 1),
        'status': 'completed',
        **variant_flags,
    }
    print(f'    Done in {elapsed:.1f}s | final r={final_corr:.6f}\n')
    return result

## 3. Phase A: Structural Test (spacing=1.0)

All runs use `a=1.0`. At this spacing, displacement vectors equal unit vectors
and `a^d = 1`. This tests the structural difference of having 2 vs 3 spacetime
node features (removing the constant `a=1` column).

In [ ]:
phase_a_results = []

print('=' * 60)
print('PHASE A: Structural test (spacing=1.0)')
print('=' * 60)

for variant_name, flags in VARIANTS.items():
    for size in LATTICE_SIZES:
        result = train_variant(
            variant_name=variant_name,
            variant_flags=flags,
            lattice_size=size,
            spacing=1.0,
            action_scale=1.0,
            phase='a',
        )
        phase_a_results.append(result)

# Save incrementally
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_BASE / 'phase_a_results.json', 'w') as f:
    json.dump(phase_a_results, f, indent=2)
print(f'Phase A results saved to {OUTPUT_BASE / "phase_a_results.json"}')

## 4. Phase B: Scaling Test (spacing=1/L)

Each lattice size uses `a = 1/L`, making displacement vectors and `a^d`
non-trivial. Target actions are rescaled by `(1/L)^d` to match.

In [ ]:
phase_b_results = []

print('=' * 60)
print('PHASE B: Scaling test (spacing=1/L)')
print('=' * 60)

for variant_name, flags in VARIANTS.items():
    for size in LATTICE_SIZES:
        spacing = 1.0 / size
        action_scale = spacing ** 2  # d=2
        result = train_variant(
            variant_name=variant_name,
            variant_flags=flags,
            lattice_size=size,
            spacing=spacing,
            action_scale=action_scale,
            phase='b',
        )
        phase_b_results.append(result)

with open(OUTPUT_BASE / 'phase_b_results.json', 'w') as f:
    json.dump(phase_b_results, f, indent=2)
print(f'Phase B results saved to {OUTPUT_BASE / "phase_b_results.json"}')

## 5. Combined Results & Analysis

In [ ]:
all_results = phase_a_results + phase_b_results

# Save combined
with open(OUTPUT_BASE / 'summary_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

# Print tables
for phase, phase_results in [('A', phase_a_results), ('B', phase_b_results)]:
    completed = [r for r in phase_results if r.get('status') == 'completed']
    if not completed:
        continue
    print(f'\n{"=" * 80}')
    print(f'PHASE {phase} RESULTS')
    print(f'{"=" * 80}')
    print(f'{"Variant":<18} {"Size":<8} {"Spacing":<10} {"Corr (r)":<12} {"Rel Err":<12} {"Time (s)":<10}')
    print(f'{"-" * 80}')
    for r in completed:
        print(
            f'{r["variant"]:<18} '
            f'{r["lattice_size"]}x{r["lattice_size"]:<4} '
            f'{r["spacing"]:<10.4f} '
            f'{r["final_val_corr"]:<12.6f} '
            f'{r["final_rel_err"]:<12.6f} '
            f'{r["training_time_s"]:<10.1f}'
        )

In [ ]:
# === Visualization ===
def plot_phase_results(phase_results, phase_label):
    completed = [r for r in phase_results if r.get('status') == 'completed']
    if not completed:
        print(f'No completed results for phase {phase_label}')
        return

    variant_names = list(VARIANTS.keys())
    colors = {'baseline': '#cc4444', 'a_edges_vol': '#4488ff',
              'a_edges_no_vol': '#44bb88', 'a_nodes_no_vol': '#cc44ff'}
    markers = {'baseline': 'o', 'a_edges_vol': 's',
               'a_edges_no_vol': 'D', 'a_nodes_no_vol': '^'}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for vname in variant_names:
        vr = [r for r in completed if r['variant'] == vname]
        if not vr:
            continue
        sizes = [r['lattice_size'] for r in vr]
        corrs = [r['final_val_corr'] for r in vr]
        errs = [r['final_rel_err'] for r in vr]

        axes[0].plot(sizes, corrs, f'-{markers[vname]}', color=colors[vname],
                     label=vname, markersize=8, alpha=0.85)
        axes[1].plot(sizes, errs, f'-{markers[vname]}', color=colors[vname],
                     label=vname, markersize=8, alpha=0.85)

    axes[0].set_xlabel('Lattice Size (L)')
    axes[0].set_ylabel('Pearson r')
    axes[0].set_title(f'Phase {phase_label}: Energy Correlation')
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.2)
    axes[0].set_xticks(LATTICE_SIZES)

    axes[1].set_xlabel('Lattice Size (L)')
    axes[1].set_ylabel('Relative Error')
    axes[1].set_title(f'Phase {phase_label}: Relative Error')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.2)
    axes[1].set_xticks(LATTICE_SIZES)
    axes[1].set_yscale('log')

    plt.suptitle(f'Displacement Edge Experiment \u2014 Phase {phase_label}', fontsize=13)
    plt.tight_layout()
    plt.show()

plot_phase_results(phase_a_results, 'A (spacing=1.0)')
plot_phase_results(phase_b_results, 'B (spacing=1/L)')

In [ ]:
# === Side-by-side Phase A vs Phase B for each variant ===
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)

variant_names = list(VARIANTS.keys())
phase_colors = {'a': '#4488ff', 'b': '#cc4444'}

for idx, vname in enumerate(variant_names):
    ax = axes[idx]
    for phase, phase_results in [('a', phase_a_results), ('b', phase_b_results)]:
        vr = [r for r in phase_results if r.get('status') == 'completed' and r['variant'] == vname]
        if not vr:
            continue
        sizes = [r['lattice_size'] for r in vr]
        corrs = [r['final_val_corr'] for r in vr]
        label = f'Phase {phase.upper()} ({"a=1" if phase=="a" else "a=1/L"})'
        ax.plot(sizes, corrs, '-o', color=phase_colors[phase], label=label, markersize=7)

    ax.set_title(vname, fontsize=11)
    ax.set_xlabel('Lattice Size (L)')
    ax.set_xticks(LATTICE_SIZES)
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=8)

axes[0].set_ylabel('Pearson r')
plt.suptitle('Phase A vs Phase B: Per-Variant Comparison', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Summary

Results are saved to `experiments/runs/displacement_edges/`:
- `phase_a_results.json` — structural test
- `phase_b_results.json` — scaling test
- `summary_results.json` — combined
- `phase_{a,b}/{variant}_{L}x{L}/model_final.pt` — checkpoints